In [13]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [14]:
import logging
import pandas as pd
import numpy as np
import awswrangler as wr
from matplotlib import pyplot as plt
import seaborn as sns
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [15]:
#Lillydoo - 68c2d9007bd2ec4485bb98ed
# Asambeauty - 682ed8362fc068cde38c3dff
# Störtebekker - 69fc8fed886052eb05c43ac5
# Junglueck - 682b1d8a1de6c855565aeff2
# Plantura - 69a9ac27b59e28048ab075a5
conversion_action_id = "69403af0a5250a665df6883f" #Nikin: 6834787b13526dc3d1017e06"
customer = "LILLYDOO"
start_date = "20250601"
end_date = "20260602"
date_range = pd.date_range(start=start_date, end=end_date, freq="D").strftime("%Y%m%d").tolist()
url = return_api_url()
print(f"url = {url}")
workspace_ids = return_workspace_ids()
workspace_id = [acc["id"] for acc in workspace_ids if acc["name"] == customer]
workspace_id = workspace_id[0]

In [16]:
data_file_path = f"DataChecks/targeting_history_ga_conversion_update/data/targeting_history_{customer}_{conversion_action_id}_{start_date}_{end_date}_strategies.csv"
try:
    df = pd.read_csv(data_file_path)
except FileNotFoundError:
    print("File not found, creating new DataFrame.")
    df = pd.DataFrame()
    for date in date_range:
        try:
            print(f"Reading data for {date}")
            path = f"s3://{workspace_id}/targeting.history/{date}/ga_conversion_update_{conversion_action_id}.parquet"
            temp = wr.s3.read_parquet(path)
            temp = temp[["created","strategy"]].drop_duplicates()
        except wr.exceptions.NoFilesFound:
            print(f". No data for {date}")
            continue
        temp["bucket_date"] = date
        df = pd.concat([df, temp])
df.to_csv(data_file_path)

In [17]:
df